# s6s7figure.ipynb

Exports Supplement Figures S6 and S7.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


# Supplementary Figure 6: regional sample-count maps

Regional versions of the two map panels in `s1figure.ipynb`: valid monthly MLD counts and total profile counts.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.ticker as ticker
from matplotlib import font_manager
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Register and verify the project-local Times New Roman fonts.
times_font_dir = Path(
    configured_path('font_dir')
)
times_font_files = [
    times_font_dir / 'Times New Roman.ttf',
    times_font_dir / 'Times New Roman Bold.ttf',
]
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(
    font_manager.findfont('Times New Roman', fallback_to_default=False)
).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')

mpl.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})
print(f'Times New Roman: {resolved_times}')

In [ ]:
base_dir = Path(configured_path('work_dir'))
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
obs_path = Path(configured_path('wod_monthly_mld_file'))
profile_path = Path(configured_path('work_dir')) / 'MLD_profile_counts_1982_2023.nc'
mask_path = Path(configured_path('reccap_mask_file'))

with xr.open_dataset(obs_path) as ds_obs:
    valid_month_count = ds_obs['MLD_d003'].notnull().sum('time').load()
with xr.open_dataset(profile_path) as ds_profiles:
    total_profile_count = ds_profiles['profile_count_all'].sum('time').load()
with xr.open_dataset(mask_path) as ds_mask:
    open_ocean = ds_mask['open_ocean'].load()

# Profile-count longitudes are stored as [-180, 180); align them with the 0--360 mask.
total_profile_count = total_profile_count.assign_coords(
    lon=(total_profile_count['lon'] % 360)
).sortby('lon')
valid_month_count = valid_month_count.where(open_ocean).where(valid_month_count > 0)
total_profile_count = total_profile_count.where(open_ocean).where(total_profile_count > 0)

fields = [
    (valid_month_count, 'Valid MLD sample count', [0, 25, 50, 75, 100, 125, 150, 175, 200]),
    (total_profile_count, 'Profile count of all', [0, 50, 100, 200, 300, 500, 750, 1000, 1500]),
]
data_crs = ccrs.PlateCarree()
cmap = plt.cm.viridis

In [ ]:
def add_map_contents(ax, field, title, levels, panel_label):
    # Duplicate the first longitude column at +360° to avoid a polar-map seam.
    cyclic = field.isel(lon=0).assign_coords(lon=field['lon'].isel(lon=0) + 360)
    field_plot = xr.concat([field, cyclic], dim='lon')
    contour = field_plot.plot.contourf(
        ax=ax, transform=data_crs, levels=levels, cmap=cmap,
        add_colorbar=False, add_labels=False, extend='max', zorder=1,
    )
    ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='none', zorder=3)
    ax.coastlines(linewidth=0.55, color='0.2', zorder=4)
    ax.set_title(title, fontsize=20, pad=8)
    ax.text(
        0.015, 1.025, f'({panel_label})', transform=ax.transAxes,
        fontsize=22, fontweight='bold', ha='left', va='bottom', clip_on=False,
    )
    return contour


def add_horizontal_colorbar(fig, ax, contour, levels):
    cbar = fig.colorbar(
        contour, ax=ax, orientation='horizontal', ticks=levels,
        shrink=0.86, pad=0.075, fraction=0.055, aspect=32,
    )
    cbar.ax.tick_params(labelsize=13, length=4)
    return cbar


def polar_circle_boundary(ax):
    theta = np.linspace(0, 2 * np.pi, 361)
    vertices = np.column_stack([0.5 + 0.5 * np.cos(theta), 0.5 + 0.5 * np.sin(theta)])
    ax.set_boundary(mpath.Path(vertices), transform=ax.transAxes)


def projected_region_boundary(projection, lon_min, lon_max, lat_min, lat_max, n=400):
    lons = np.concatenate([
        np.linspace(lon_min, lon_max, n),
        np.full(n, lon_max),
        np.linspace(lon_max, lon_min, n),
        np.full(n, lon_min),
    ])
    lats = np.concatenate([
        np.full(n, lat_min),
        np.linspace(lat_min, lat_max, n),
        np.full(n, lat_max),
        np.linspace(lat_max, lat_min, n),
    ])
    vertices = projection.transform_points(data_crs, lons, lats)[:, :2]
    vertices = np.vstack([vertices, vertices[0]])
    codes = np.full(len(vertices), mpath.Path.LINETO, dtype=np.uint8)
    codes[0] = mpath.Path.MOVETO
    codes[-1] = mpath.Path.CLOSEPOLY
    return mpath.Path(vertices, codes)

In [ ]:
# Southern Ocean: all longitudes south of 40°S.
# Use a narrower color range here to reveal more regional spatial detail.
southern_fields = [
    (valid_month_count, 'Valid MLD sample count', [0, 25, 50, 75, 100]),
    (total_profile_count, 'Profile count of all', [0, 50, 100, 200, 300, 500, 750, 1000]),
]
south_projection = ccrs.SouthPolarStereo(central_longitude=0)
fig, axes = plt.subplots(
    1, 2, figsize=(12.5, 6.4), dpi=200,
    subplot_kw={'projection': south_projection},
)
fig.subplots_adjust(left=0.025, right=0.985, bottom=0.17, top=0.91, wspace=0.075)

for ax, (field, title, levels), label in zip(axes, southern_fields, ['a', 'b']):
    ax.set_extent([-180, 180, -90, -40], crs=data_crs)
    polar_circle_boundary(ax)
    contour = add_map_contents(ax, field.sel(lat=slice(-90, -40)), title, levels, label)
    gl = ax.gridlines(
        crs=data_crs, linewidth=0.45, color='0.3', alpha=0.65, linestyle='--',
        xlocs=np.arange(-180, 181, 45), ylocs=[-80, -60, -40], draw_labels=False,
    )
    add_horizontal_colorbar(fig, ax, contour, levels)

fig.savefig(
    output_dir / 's7figure.png', dpi=600,
    bbox_inches='tight', pad_inches=0, facecolor='white',
)
plt.show()

In [ ]:
# North Atlantic sector: 70°W--5°W, 52°N--68°N.
lon_min, lon_max = -70, -5
lat_min, lat_max = 52, 68
north_projection = ccrs.NorthPolarStereo(central_longitude=-37.5)
fan_boundary = projected_region_boundary(
    north_projection, lon_min, lon_max, lat_min, lat_max
)

fig, axes = plt.subplots(
    1, 2, figsize=(13.5, 5.9), dpi=200,
    subplot_kw={'projection': north_projection},
)
fig.subplots_adjust(left=0.025, right=0.985, bottom=0.18, top=0.90, wspace=0.055)

for ax, (field, title, levels), label in zip(axes, fields, ['a', 'b']):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=data_crs)
    ax.set_boundary(fan_boundary, transform=ax.transData)
    for spine in ax.spines.values():
        spine.set_visible(False)
    contour = add_map_contents(ax, field.sel(lat=slice(lat_min, lat_max)), title, levels, label)
    ax.gridlines(
        crs=data_crs, linewidth=0.45, color='0.3', alpha=0.65, linestyle='--',
        xlocs=np.arange(-70, -4, 10), ylocs=np.arange(52, 69, 4), draw_labels=False,
    )
    add_horizontal_colorbar(fig, ax, contour, levels)

fig.savefig(
    output_dir / 's6figure.png', dpi=600,
    bbox_inches='tight', pad_inches=0, facecolor='white',
)
plt.show()